In [2]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print


# =========================================================
# 0. Parameters
# =========================================================
CSV_PATH = "0050_OHLCV.csv"
INITIAL_CAPITAL = 1_000_000
TRADING_DAYS = 252

BUY_FEE = 0.001425
SELL_FEE = 0.001425
SELL_TAX = 0.003
LOT_SHARES = 1000

# Squeeze / indicator parameters
LENGTH = 20
MULT = 2.0
LENGTH_KC = 20
MULT_KC = 1.5
ATR_PERIOD = 14

# Position sizing
SPEC_INITIAL_ENTRY_FRACTION = 0.25      # Squeeze ON: earlier / speculative
INVEST_INITIAL_ENTRY_FRACTION = 0.50    # Squeeze OFF: confirmed / trend
ADDON_LOTS = 1
ADDON_MIN_DISTANCE_ATR = 1.0

# ON bucket stricter exits
ON_FIXED_TP = 0.20
ON_FIXED_SL = -0.15
ON_YOYO_ATR = 4.0
ON_CHAND_ATR = 4.0
ON_BE_INIT_ATR = 2.0
ON_BE_TRIGGER_ATR = 2.0
ON_BE_TRAIL_ATR = 3.0
ON_EMA_FAST = 8
ON_EMA_SLOW = 21

# OFF bucket corresponding, looser / trend exits
OFF_FIXED_TP = 0.35
OFF_FIXED_SL = -0.18
OFF_YOYO_ATR = 5.0
OFF_CHAND_ATR = 5.0
OFF_BE_INIT_ATR = 2.5
OFF_BE_TRIGGER_ATR = 2.0
OFF_BE_TRAIL_ATR_EARLY = 5.0
OFF_BE_TRAIL_ATR_LATE = 4.0
OFF_LOCK_PROFIT_RETURN = 0.20
OFF_LOCK_PROFIT_FLOOR = 0.05
OFF_TIGHTEN_RETURN = 0.35
OFF_EMA_FAST = 13
OFF_EMA_SLOW = 34


EXIT_PROFILES = {
    "hold": "Buckets - Hold to End",
    "fixed": "Buckets - ON Fixed 20/-15 vs OFF Fixed 35/-18",
    "atr_yoyo": "Buckets - ON 4ATR Yo-Yo vs OFF 5ATR Yo-Yo",
    "chandelier": "Buckets - ON Chandelier 4ATR vs OFF Chandelier 5ATR",
    "breakeven": "Buckets - ON Breakeven 3ATR vs OFF Trend Breakeven 5/4ATR",
    "ema_cross": "Buckets - ON 8/21 EMA vs OFF 13/34 EMA",
}


# =========================================================
# 1. Load data
# =========================================================
def load_data(csv_path=CSV_PATH):
    df = pd.read_csv(csv_path)
    rename_map = {
        "年月日": "date",
        "日期": "date",
        "Date": "date",
        "date": "date",
        "開盤價(元)": "open",
        "最高價(元)": "high",
        "最低價(元)": "low",
        "收盤價(元)": "close",
        "成交量(千股)": "volume",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
        "撟湔??功": "date",
        "?交?": "date",
        "?????": "open",
        "?擃(??": "high",
        "?雿(??": "low",
        "?嗥????": "close",
        "?漱???)": "volume",
    }
    df = df.rename(columns=rename_map)

    needed = ["date", "open", "high", "low", "close"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Current columns: {df.columns.tolist()}")

    df["date"] = pd.to_datetime(df["date"])
    for c in ["open", "high", "low", "close", "volume"]:
        if c in df.columns:
            df[c] = (
                df[c].astype(str)
                .str.replace(",", "", regex=False)
                .replace("nan", np.nan)
            )
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "volume" not in df.columns:
        df["volume"] = 0

    df = df.set_index("date").sort_index()
    return df.dropna(subset=["open", "high", "low", "close"])


# =========================================================
# 2. Indicators and entry signals
# =========================================================
def true_range(df):
    prev_close = df["close"].shift(1)
    tr1 = df["high"] - df["low"]
    tr2 = (df["high"] - prev_close).abs()
    tr3 = (df["low"] - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


def wilder_rma(series, length):
    values = series.astype(float).values
    out = np.full(len(values), np.nan, dtype=float)
    if len(values) < length:
        return pd.Series(out, index=series.index)
    out[length - 1] = np.nanmean(values[:length])
    alpha = 1.0 / length
    for i in range(length, len(values)):
        if np.isnan(out[i - 1]) or np.isnan(values[i]):
            out[i] = np.nan
        else:
            out[i] = out[i - 1] + alpha * (values[i] - out[i - 1])
    return pd.Series(out, index=series.index)


def rolling_linreg_last(y, window):
    x = np.arange(window)
    x_mean = x.mean()
    denom = ((x - x_mean) ** 2).sum()

    def calc(arr):
        if np.isnan(arr).any():
            return np.nan
        y_mean = arr.mean()
        slope = ((x - x_mean) * (arr - y_mean)).sum() / denom
        intercept = y_mean - slope * x_mean
        return intercept + slope * (window - 1)

    return y.rolling(window).apply(calc, raw=True)


def add_indicators_and_entries(df):
    out = df.copy()
    close = out["close"]
    high = out["high"]
    low = out["low"]

    basis = close.rolling(LENGTH).mean()
    dev = MULT * close.rolling(LENGTH).std(ddof=0)
    out["upperBB"] = basis + dev
    out["lowerBB"] = basis - dev

    ma = close.rolling(LENGTH_KC).mean()
    tr = true_range(out)
    range_ma = tr.rolling(LENGTH_KC).mean()
    out["upperKC"] = ma + range_ma * MULT_KC
    out["lowerKC"] = ma - range_ma * MULT_KC

    out["sqz_on"] = (out["lowerBB"] > out["lowerKC"]) & (out["upperBB"] < out["upperKC"])
    out["sqz_off"] = (out["lowerBB"] < out["lowerKC"]) & (out["upperBB"] > out["upperKC"])

    highest_high = high.rolling(LENGTH_KC).max()
    lowest_low = low.rolling(LENGTH_KC).min()
    mean_hl = (highest_high + lowest_low) / 2
    mean_all = (mean_hl + ma) / 2
    momentum_input = close - mean_all
    out["val"] = rolling_linreg_last(momentum_input, LENGTH_KC)
    out["val_prev"] = out["val"].shift(1)

    # 0518_1: squeeze-off confirmed trend entry.
    out["entry_signal_0518_off"] = (
        out["sqz_on"].shift(1).fillna(False)
        & out["sqz_off"].fillna(False)
        & (out["val"] > 0)
        & (out["val"] > out["val_prev"])
    )

    # 0050_1: squeeze-on early/speculative entry.
    mom = out["val"]
    out["sqz_on_3"] = (
        out["sqz_on"].fillna(False)
        & out["sqz_on"].shift(1).fillna(False)
        & out["sqz_on"].shift(2).fillna(False)
    )
    out["mom_positive_continue"] = (mom > 0) & (mom.shift(1) > 0) & (mom.shift(2) > 0)
    out["mom_negative_improving"] = (
        (mom < 0)
        & (mom.shift(1) < 0)
        & (mom.shift(2) < 0)
        & (mom.shift(2) < mom.shift(1))
        & (mom.shift(1) < mom)
    )
    out["mom_negative_to_positive"] = (mom.shift(2) < 0) & (mom.shift(1) > 0) & (mom > 0)
    out["entry_signal_0050_on"] = out["sqz_on_3"] & (
        out["mom_positive_continue"]
        | out["mom_negative_improving"]
        | out["mom_negative_to_positive"]
    )

    out["tr"] = tr
    out["atr14"] = wilder_rma(tr, ATR_PERIOD)
    out["atr14_prev"] = out["atr14"].shift(1)

    out["ema8"] = close.ewm(span=ON_EMA_FAST, adjust=False).mean()
    out["ema21"] = close.ewm(span=ON_EMA_SLOW, adjust=False).mean()
    out["ema8_21_death"] = (out["ema8"].shift(1) >= out["ema21"].shift(1)) & (out["ema8"] < out["ema21"])

    out["ema13"] = close.ewm(span=OFF_EMA_FAST, adjust=False).mean()
    out["ema34"] = close.ewm(span=OFF_EMA_SLOW, adjust=False).mean()
    out["ema13_34_death"] = (out["ema13"].shift(1) >= out["ema34"].shift(1)) & (out["ema13"] < out["ema34"])
    return out


# =========================================================
# 3. Exit helpers
# =========================================================
def resolve_stop_tp_exit(open_price, high_price, low_price, stop_price, tp_price, prefix):
    if open_price <= stop_price:
        return open_price, f"{prefix}_stop_gap_open"
    if open_price >= tp_price:
        return open_price, f"{prefix}_tp_gap_open"

    hit_stop = low_price <= stop_price
    hit_tp = high_price >= tp_price
    if hit_stop and hit_tp:
        return stop_price, f"{prefix}_same_bar_both_hit_conservative_stop"
    if hit_stop:
        return stop_price, f"{prefix}_stop_intraday"
    if hit_tp:
        return tp_price, f"{prefix}_tp_intraday"
    return None, None


def resolve_stop_only_exit(open_price, low_price, stop_price, prefix):
    if open_price <= stop_price:
        return open_price, f"{prefix}_gap_open"
    if low_price <= stop_price:
        return stop_price, f"{prefix}_intraday"
    return None, None


def buy_cost(price, qty_shares):
    gross = qty_shares * price
    fee = gross * BUY_FEE
    return gross, fee, gross + fee


def affordable_lots(cash, price):
    _, _, one_lot_cost = buy_cost(price, LOT_SHARES)
    return 0 if one_lot_cost <= 0 else int(cash // one_lot_cost)


def initial_lots(affordable, fraction):
    if affordable <= 0:
        return 0
    return min(max(1, int(np.floor(affordable * fraction))), affordable)


def new_bucket(label, entry_type, sizing_fraction):
    return {
        "label": label,
        "entry_type": entry_type,
        "sizing_fraction": sizing_fraction,
        "shares": 0.0,
        "avg_entry_price": np.nan,
        "avg_entry_atr": np.nan,
        "base_entry_price": np.nan,
        "base_entry_atr": np.nan,
        "highest_high": np.nan,
        "highest_close": np.nan,
        "trail_stop": np.nan,
        "current_cycle": None,
        "pending_entry_count": 0,
        "pending_ema_exit": False,
        "cycles": [],
        "executed_entries": 0,
        "total_units_bought": 0.0,
        "completed_cycles": 0,
    }


def active_stop_for_addon(bucket, atr_ref, exit_profile):
    if bucket["shares"] <= 0 or np.isnan(atr_ref) or exit_profile == "hold":
        return np.nan
    base = bucket["base_entry_price"]
    high_close = bucket["highest_close"]
    if np.isnan(base) or np.isnan(high_close):
        return np.nan

    if exit_profile == "fixed":
        return base * (1 + (ON_FIXED_SL if bucket["entry_type"] == "squeeze_on" else OFF_FIXED_SL))
    if exit_profile == "atr_yoyo":
        mult = ON_YOYO_ATR if bucket["entry_type"] == "squeeze_on" else OFF_YOYO_ATR
        return bucket["highest_high"] - mult * atr_ref
    if exit_profile == "chandelier":
        mult = ON_CHAND_ATR if bucket["entry_type"] == "squeeze_on" else OFF_CHAND_ATR
        return high_close - mult * atr_ref
    if exit_profile == "breakeven":
        return breakeven_stop(bucket, atr_ref)
    return np.nan


def breakeven_stop(bucket, atr_ref):
    base = bucket["base_entry_price"]
    base_atr = bucket["base_entry_atr"]
    high_close = bucket["highest_close"]
    if np.isnan(base) or np.isnan(base_atr) or np.isnan(high_close) or np.isnan(atr_ref):
        return np.nan

    if bucket["entry_type"] == "squeeze_on":
        initial_stop = base - ON_BE_INIT_ATR * base_atr
        trigger = base + ON_BE_TRIGGER_ATR * base_atr
        raw_stop = max(base, high_close - ON_BE_TRAIL_ATR * atr_ref) if high_close >= trigger else initial_stop
    else:
        initial_stop = base - OFF_BE_INIT_ATR * base_atr
        trigger = base + OFF_BE_TRIGGER_ATR * base_atr
        unrealized_return = high_close / base - 1
        trail_mult = OFF_BE_TRAIL_ATR_LATE if unrealized_return >= OFF_TIGHTEN_RETURN else OFF_BE_TRAIL_ATR_EARLY
        raw_stop = high_close - trail_mult * atr_ref
        raw_stop = max(raw_stop, base) if high_close >= trigger else min(raw_stop, initial_stop)
        if unrealized_return >= OFF_LOCK_PROFIT_RETURN:
            raw_stop = max(raw_stop, base * (1 + OFF_LOCK_PROFIT_FLOOR))

    return raw_stop if np.isnan(bucket["trail_stop"]) else max(bucket["trail_stop"], raw_stop)


def can_addon(bucket, entry_price, atr_ref, exit_profile):
    if bucket["shares"] <= 0:
        return True
    active_stop = active_stop_for_addon(bucket, atr_ref, exit_profile)
    if np.isnan(active_stop) or np.isnan(atr_ref):
        return True
    return (entry_price - active_stop) >= ADDON_MIN_DISTANCE_ATR * atr_ref


# =========================================================
# 4. Two-bucket backtest for one exit profile
# =========================================================
def backtest_buckets(df, exit_profile, initial_capital=INITIAL_CAPITAL):
    out = df.copy()
    cash = float(initial_capital)
    total_cost = 0.0

    spec = new_bucket("Squeeze ON bucket", "squeeze_on", SPEC_INITIAL_ENTRY_FRACTION)
    invest = new_bucket("Squeeze OFF bucket", "squeeze_off", INVEST_INITIAL_ENTRY_FRACTION)
    buckets = [invest, spec]

    equity_list, cash_list, spec_shares_list, invest_shares_list, total_cost_list = [], [], [], [], []

    def execute_buy(bucket, dt, price, atr_ref, qty_shares):
        nonlocal cash, total_cost
        if qty_shares <= 0:
            return False
        gross, fee, total_required = buy_cost(price, qty_shares)
        if total_required > cash:
            return False

        prev_shares = bucket["shares"]
        new_shares = prev_shares + qty_shares
        cash -= total_required
        total_cost += fee
        bucket["executed_entries"] += 1
        bucket["total_units_bought"] += qty_shares

        if prev_shares == 0:
            bucket["avg_entry_price"] = price
            bucket["avg_entry_atr"] = atr_ref
            bucket["base_entry_price"] = price
            bucket["base_entry_atr"] = atr_ref
            bucket["highest_high"] = price
            bucket["highest_close"] = price
            bucket["trail_stop"] = np.nan
            bucket["current_cycle"] = {
                "exit_profile": exit_profile,
                "strategy_bucket": bucket["label"],
                "entry_type": bucket["entry_type"],
                "start_time": dt,
                "entries": 1,
                "units_bought": qty_shares,
                "gross_entry_value": gross,
                "entry_fees": fee,
            }
        else:
            bucket["avg_entry_price"] = (bucket["avg_entry_price"] * prev_shares + price * qty_shares) / new_shares
            if np.isnan(bucket["avg_entry_atr"]):
                bucket["avg_entry_atr"] = atr_ref
            elif not np.isnan(atr_ref):
                bucket["avg_entry_atr"] = (bucket["avg_entry_atr"] * prev_shares + atr_ref * qty_shares) / new_shares
            bucket["current_cycle"]["entries"] += 1
            bucket["current_cycle"]["units_bought"] += qty_shares
            bucket["current_cycle"]["gross_entry_value"] += gross
            bucket["current_cycle"]["entry_fees"] += fee

        bucket["shares"] = new_shares
        return True

    def close_bucket(bucket, dt, price, reason):
        nonlocal cash, total_cost
        shares = bucket["shares"]
        if shares <= 0:
            return
        gross_exit = shares * price
        sell_fee = gross_exit * SELL_FEE
        sell_tax = gross_exit * SELL_TAX
        net_exit = gross_exit - sell_fee - sell_tax
        total_cost += sell_fee + sell_tax
        cash += net_exit

        cycle = bucket["current_cycle"]
        cycle_cost = cycle["gross_entry_value"] + cycle["entry_fees"]
        cycle_pnl = net_exit - cycle_cost
        cycle_return = cycle_pnl / cycle_cost if cycle_cost else np.nan
        bucket["cycles"].append({
            **cycle,
            "end_time": dt,
            "avg_entry_price": bucket["avg_entry_price"],
            "base_entry_price": bucket["base_entry_price"],
            "exit_price": price,
            "pnl": cycle_pnl,
            "net_return": cycle_return,
            "exit_reason": reason,
            "exit_fee_tax": sell_fee + sell_tax,
        })

        bucket["shares"] = 0.0
        bucket["avg_entry_price"] = np.nan
        bucket["avg_entry_atr"] = np.nan
        bucket["base_entry_price"] = np.nan
        bucket["base_entry_atr"] = np.nan
        bucket["highest_high"] = np.nan
        bucket["highest_close"] = np.nan
        bucket["trail_stop"] = np.nan
        bucket["current_cycle"] = None
        bucket["completed_cycles"] += 1

    for i in range(len(out)):
        row = out.iloc[i]
        dt = out.index[i]
        open_price = float(row["open"])
        high_price = float(row["high"])
        low_price = float(row["low"])
        close_price = float(row["close"])
        atr_prev = row["atr14_prev"]

        # Pending EMA exits: signal known after previous close, executed at current open.
        for bucket in buckets:
            if bucket["pending_ema_exit"] and bucket["shares"] > 0:
                close_bucket(bucket, dt, open_price, f"{bucket['entry_type']}_{exit_profile}_next_open_exit")
            bucket["pending_ema_exit"] = False

        # Pending entries: signal known after previous close, executed at current open.
        for bucket in buckets:
            if bucket["pending_entry_count"] > 0 and not np.isnan(atr_prev):
                if bucket["shares"] == 0:
                    affordable = affordable_lots(cash, open_price)
                    lots = initial_lots(affordable, bucket["sizing_fraction"])
                    execute_buy(bucket, dt, open_price, atr_prev, lots * LOT_SHARES)
                elif can_addon(bucket, open_price, atr_prev, exit_profile):
                    for _ in range(bucket["pending_entry_count"]):
                        execute_buy(bucket, dt, open_price, atr_prev, ADDON_LOTS * LOT_SHARES)
                bucket["pending_entry_count"] = 0

        # Intraday exits. Stops use prior ATR and prior high/close anchors.
        for bucket in buckets:
            if bucket["shares"] <= 0 or exit_profile in {"hold", "ema_cross"}:
                continue

            if exit_profile == "fixed":
                if bucket["entry_type"] == "squeeze_on":
                    stop = bucket["avg_entry_price"] * (1 + ON_FIXED_SL)
                    tp = bucket["avg_entry_price"] * (1 + ON_FIXED_TP)
                    prefix = "squeeze_on_fixed"
                else:
                    stop = bucket["avg_entry_price"] * (1 + OFF_FIXED_SL)
                    tp = bucket["avg_entry_price"] * (1 + OFF_FIXED_TP)
                    prefix = "squeeze_off_fixed"
                exit_price, reason = resolve_stop_tp_exit(open_price, high_price, low_price, stop, tp, prefix)
                if exit_price is not None:
                    close_bucket(bucket, dt, exit_price, reason)

            elif exit_profile == "atr_yoyo" and not np.isnan(atr_prev):
                mult = ON_YOYO_ATR if bucket["entry_type"] == "squeeze_on" else OFF_YOYO_ATR
                tp_pct = ON_FIXED_TP if bucket["entry_type"] == "squeeze_on" else OFF_FIXED_TP
                raw_stop = bucket["highest_high"] - mult * atr_prev
                bucket["trail_stop"] = raw_stop if np.isnan(bucket["trail_stop"]) else max(bucket["trail_stop"], raw_stop)
                tp = bucket["avg_entry_price"] * (1 + tp_pct)
                exit_price, reason = resolve_stop_tp_exit(open_price, high_price, low_price, bucket["trail_stop"], tp, f"{bucket['entry_type']}_yoyo")
                if exit_price is not None:
                    close_bucket(bucket, dt, exit_price, reason)

            elif exit_profile == "chandelier" and not np.isnan(atr_prev):
                mult = ON_CHAND_ATR if bucket["entry_type"] == "squeeze_on" else OFF_CHAND_ATR
                raw_stop = bucket["highest_close"] - mult * atr_prev
                bucket["trail_stop"] = raw_stop if np.isnan(bucket["trail_stop"]) else max(bucket["trail_stop"], raw_stop)
                exit_price, reason = resolve_stop_only_exit(open_price, low_price, bucket["trail_stop"], f"{bucket['entry_type']}_chandelier")
                if exit_price is not None:
                    close_bucket(bucket, dt, exit_price, reason)

            elif exit_profile == "breakeven" and not np.isnan(atr_prev):
                stop = breakeven_stop(bucket, atr_prev)
                if not np.isnan(stop):
                    bucket["trail_stop"] = stop if np.isnan(bucket["trail_stop"]) else max(bucket["trail_stop"], stop)
                    exit_price, reason = resolve_stop_only_exit(open_price, low_price, bucket["trail_stop"], f"{bucket['entry_type']}_breakeven")
                    if exit_price is not None:
                        close_bucket(bucket, dt, exit_price, reason)

        for bucket in buckets:
            if bucket["shares"] > 0:
                bucket["highest_high"] = max(bucket["highest_high"], high_price) if not np.isnan(bucket["highest_high"]) else high_price
                bucket["highest_close"] = max(bucket["highest_close"], close_price) if not np.isnan(bucket["highest_close"]) else close_price

        equity = cash + sum(bucket["shares"] * close_price for bucket in buckets)
        equity_list.append(equity)
        cash_list.append(cash)
        invest_shares_list.append(invest["shares"])
        spec_shares_list.append(spec["shares"])
        total_cost_list.append(total_cost)

        if i < len(out) - 1:
            if bool(row["entry_signal_0518_off"]):
                invest["pending_entry_count"] += 1
            if bool(row["entry_signal_0050_on"]):
                spec["pending_entry_count"] += 1

            if exit_profile == "ema_cross":
                if invest["shares"] > 0 and bool(row["ema13_34_death"]):
                    invest["pending_ema_exit"] = True
                if spec["shares"] > 0 and bool(row["ema8_21_death"]):
                    spec["pending_ema_exit"] = True

    # Liquidate final open positions only for metrics and completed cycle records.
    final_dt = out.index[-1]
    final_close = float(out["close"].iloc[-1])
    for bucket in buckets:
        if bucket["shares"] > 0:
            close_bucket(bucket, final_dt, final_close, f"{bucket['entry_type']}_{exit_profile}_final_liquidation")

    final_equity_liquidated = cash

    out["cash"] = cash_list
    out["invest_shares"] = invest_shares_list
    out["spec_shares"] = spec_shares_list
    out["shares"] = out["invest_shares"] + out["spec_shares"]
    out["equity"] = equity_list
    out["equity_curve"] = out["equity"] / initial_capital
    out["strategy_return"] = out["equity"].pct_change().fillna(0.0)
    out["total_cost"] = total_cost_list

    cycles_df = pd.concat(
        [pd.DataFrame(bucket["cycles"]) for bucket in buckets if bucket["cycles"]],
        ignore_index=True,
    ) if any(bucket["cycles"] for bucket in buckets) else pd.DataFrame()

    meta = {
        "raw_squeeze_off_signals": int(out["entry_signal_0518_off"].fillna(False).sum()),
        "raw_squeeze_on_signals": int(out["entry_signal_0050_on"].fillna(False).sum()),
        "executed_entries": sum(bucket["executed_entries"] for bucket in buckets),
        "total_units_bought": sum(bucket["total_units_bought"] for bucket in buckets),
        "completed_cycles": sum(bucket["completed_cycles"] for bucket in buckets),
        "total_contribution": initial_capital,
        "final_equity_liquidated": final_equity_liquidated,
    }
    return out, cycles_df, meta


# =========================================================
# 5. Benchmark and metrics
# =========================================================
def backtest_buy_and_hold_full(df, initial_capital=INITIAL_CAPITAL):
    out = df.copy()
    entry_price = float(out["open"].iloc[0])
    shares = initial_capital / (entry_price * (1 + BUY_FEE))
    entry_fee = shares * entry_price * BUY_FEE
    out["cash"] = 0.0
    out["shares"] = shares
    out["equity"] = shares * out["close"]
    out["equity_curve"] = out["equity"] / initial_capital
    out["strategy_return"] = out["equity"].pct_change().fillna(0.0)
    out["total_cost"] = entry_fee
    final_price = float(out["close"].iloc[-1])
    gross_exit = shares * final_price
    sell_fee = gross_exit * SELL_FEE
    sell_tax = gross_exit * SELL_TAX
    final_equity = gross_exit - sell_fee - sell_tax
    cycles = pd.DataFrame([{
        "exit_profile": "benchmark",
        "strategy_bucket": "Benchmark A",
        "entry_type": "buy_and_hold",
        "start_time": out.index[0],
        "end_time": out.index[-1],
        "entries": 1,
        "units_bought": shares,
        "avg_entry_price": entry_price,
        "base_entry_price": entry_price,
        "exit_price": final_price,
        "pnl": final_equity - initial_capital,
        "net_return": final_equity / initial_capital - 1,
        "exit_reason": "buy_and_hold_full",
        "entry_fees": entry_fee,
        "exit_fee_tax": sell_fee + sell_tax,
    }])
    meta = {
        "final_equity_liquidated": final_equity,
        "executed_entries": 1,
        "total_units_bought": shares,
        "completed_cycles": 1,
        "total_contribution": initial_capital,
    }
    return out, cycles, meta


def compute_alpha_beta(strategy_bt_df, benchmark_bt_df):
    aligned = pd.concat(
        [
            strategy_bt_df["strategy_return"].rename("rp"),
            benchmark_bt_df["strategy_return"].rename("rm"),
        ],
        axis=1,
        join="inner",
    ).dropna()
    if len(aligned) < 2:
        return np.nan, np.nan, np.nan
    var_m = aligned["rm"].var(ddof=0)
    if var_m == 0 or np.isnan(var_m):
        return np.nan, np.nan, np.nan
    cov_pm = ((aligned["rp"] - aligned["rp"].mean()) * (aligned["rm"] - aligned["rm"].mean())).mean()
    beta = cov_pm / var_m
    daily_alpha = aligned["rp"].mean() - beta * aligned["rm"].mean()
    annualized_alpha = daily_alpha * TRADING_DAYS
    return beta, daily_alpha, annualized_alpha


def calculate_metrics(bt_df, cycles_df, meta, benchmark_bt_df=None):
    returns = bt_df["strategy_return"].dropna()
    final_equity = meta["final_equity_liquidated"]
    total_return = final_equity / INITIAL_CAPITAL - 1
    years = len(bt_df) / TRADING_DAYS
    annualized_return = (final_equity / INITIAL_CAPITAL) ** (1 / years) - 1
    volatility = np.sqrt(TRADING_DAYS) * returns.std(ddof=0)
    drawdown = bt_df["equity_curve"] / bt_df["equity_curve"].cummax() - 1
    sharpe = np.nan if returns.std(ddof=0) == 0 else np.sqrt(TRADING_DAYS) * returns.mean() / returns.std(ddof=0)
    beta, daily_alpha, annualized_alpha = (
        (np.nan, np.nan, np.nan) if benchmark_bt_df is None else compute_alpha_beta(bt_df, benchmark_bt_df)
    )
    return {
        "Final Equity": final_equity,
        "Total PnL": final_equity - INITIAL_CAPITAL,
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Volatility": volatility,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": drawdown.min(),
        "Win Rate": (cycles_df["net_return"] > 0).mean() if len(cycles_df) else np.nan,
        "Avg Cycle Return": cycles_df["net_return"].mean() if len(cycles_df) else np.nan,
        "Beta vs Benchmark A": beta,
        "Annualized Alpha vs Benchmark A": annualized_alpha,
        "Raw Squeeze-Off Signals": meta.get("raw_squeeze_off_signals", np.nan),
        "Raw Squeeze-On Signals": meta.get("raw_squeeze_on_signals", np.nan),
        "Executed Entries": meta.get("executed_entries", np.nan),
        "Exit Events": len(cycles_df),
        "Total Units Bought": meta.get("total_units_bought", np.nan),
        "Completed Cycles": meta.get("completed_cycles", np.nan),
        "Total Cost": bt_df["total_cost"].iloc[-1],
    }


def format_table(df):
    formatted = df.copy()
    money_cols = ["Final Equity", "Total PnL", "Total Cost"]
    pct_cols = ["Total Return", "Annualized Return", "Volatility", "Max Drawdown", "Win Rate", "Avg Cycle Return", "Annualized Alpha vs Benchmark A"]
    count_cols = ["Raw Squeeze-Off Signals", "Raw Squeeze-On Signals", "Executed Entries", "Exit Events", "Total Units Bought", "Completed Cycles"]
    for col in money_cols:
        if col in formatted:
            formatted[col] = formatted[col].map(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
    for col in pct_cols:
        if col in formatted:
            formatted[col] = formatted[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "")
    if "Sharpe Ratio" in formatted:
        formatted["Sharpe Ratio"] = formatted["Sharpe Ratio"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
    if "Beta vs Benchmark A" in formatted:
        formatted["Beta vs Benchmark A"] = formatted["Beta vs Benchmark A"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
    for col in count_cols:
        if col in formatted:
            formatted[col] = formatted[col].map(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
    return formatted


def plot_equity_curves(result_dict):
    fig, ax = plt.subplots(figsize=(16, 8))
    for label, bt_df in result_dict.items():
        ax.plot(bt_df.index, bt_df["equity"], linewidth=2, label=label)
    ax.set_title("Bucket Exit Profile Comparison")
    ax.set_ylabel("Equity")
    ax.set_xlabel("Date")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


# =========================================================
# 6. Main
# =========================================================
df = load_data()
df = add_indicators_and_entries(df)

bench_bt, bench_cycles, bench_meta = backtest_buy_and_hold_full(df)

comparison_rows = []
equity_result_dict = {"Benchmark A - Buy and Hold": bench_bt}
cycles_result_dict = {"Benchmark A - Buy and Hold": bench_cycles}

comparison_rows.append({
    "Strategy": "Benchmark A - Buy and Hold Full Capital 0050",
    **calculate_metrics(bench_bt, bench_cycles, bench_meta, benchmark_bt_df=bench_bt),
})

for profile, profile_name in EXIT_PROFILES.items():
    bt_df, cycles_df, meta = backtest_buckets(df, profile)
    equity_result_dict[profile_name] = bt_df
    cycles_result_dict[profile_name] = cycles_df
    comparison_rows.append({
        "Strategy": profile_name,
        **calculate_metrics(bt_df, cycles_df, meta, benchmark_bt_df=bench_bt),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("Final Equity", ascending=True).reset_index(drop=True)

print("===== Entry Signal Counts =====")
display(pd.DataFrame([{
    "Squeeze-Off investment signals": int(df["entry_signal_0518_off"].fillna(False).sum()),
    "Squeeze-On speculative signals": int(df["entry_signal_0050_on"].fillna(False).sum()),
    "Overlap signals": int((df["entry_signal_0518_off"].fillna(False) & df["entry_signal_0050_on"].fillna(False)).sum()),
}]))

print("===== Entry Signal Preview =====")
display(df.loc[df["entry_signal_0518_off"] | df["entry_signal_0050_on"], [
    "open", "high", "low", "close", "sqz_on", "sqz_off", "val", "val_prev",
    "entry_signal_0518_off", "entry_signal_0050_on",
]].head(60))

print("===== Raw Comparison Table =====")
display(comparison_df)

print("===== Formatted Comparison Table =====")
display(format_table(comparison_df))

print("===== Trade Cycle Summary by Strategy and Bucket =====")
all_cycles = []
for strategy_name, cycles_df in cycles_result_dict.items():
    if len(cycles_df) == 0:
        continue
    temp = cycles_df.copy()
    temp.insert(0, "Strategy", strategy_name)
    all_cycles.append(temp)

all_cycles_df = pd.concat(all_cycles, ignore_index=True) if all_cycles else pd.DataFrame()
if len(all_cycles_df):
    summary_df = (
        all_cycles_df
        .groupby(["Strategy", "strategy_bucket", "entry_type"], as_index=False)
        .agg(
            Cycles=("entry_type", "size"),
            Total_Entries=("entries", "sum"),
            Total_Units=("units_bought", "sum"),
            Avg_Return=("net_return", "mean"),
            Win_Rate=("net_return", lambda s: (s > 0).mean()),
            Total_PnL=("pnl", "sum"),
        )
    )
    display(summary_df)

    cycle_cols = [
        "exit_profile", "strategy_bucket", "entry_type", "start_time", "end_time",
        "entries", "units_bought", "avg_entry_price", "base_entry_price",
        "exit_price", "pnl", "net_return", "exit_reason", "entry_fees", "exit_fee_tax",
    ]
    for strategy_name, cycles_df in cycles_result_dict.items():
        if len(cycles_df) == 0:
            continue
        print(f"===== Trade Cycles: {strategy_name} =====")
        display(cycles_df[cycle_cols].reset_index(drop=True))

plot_equity_curves(equity_result_dict)


===== Entry Signal Counts =====


,Squeeze-Off investment signals,Squeeze-On speculative signals,Overlap signals
0,9,19,0


===== Entry Signal Preview =====


,open,high,low,close,sqz_on,sqz_off,val,val_prev,entry_signal_0518_off,entry_signal_0050_on
date,,,,,,,,,,
2016-06-29,11.6742,11.8289,11.6742,11.7743,True,False,0.002663,0.017808,False,True
2016-06-30,11.8561,11.9107,11.7652,11.9107,True,False,0.021358,0.002663,False,True
2016-07-01,11.9835,12.0836,11.9471,12.0563,False,True,0.058179,0.021358,True,False
2016-10-21,13.3842,13.4579,13.3658,13.3750,True,False,0.167458,0.180124,False,True
2016-10-24,13.4211,13.4395,13.3750,13.3934,True,False,0.162304,0.167458,False,True
2016-10-25,13.4211,13.5408,13.4211,13.5224,False,True,0.177663,0.162304,True,False
2017-09-20,15.6935,15.7221,15.6175,15.6365,True,False,0.026649,0.073235,False,True
2017-09-21,15.6079,15.7506,15.5889,15.7316,True,False,0.012680,0.026649,False,True
2017-11-10,16.0360,16.0835,15.9694,16.0835,True,False,0.056833,0.103862,False,True


===== Raw Comparison Table =====


...

===== Trade Cycles: Buckets - ON Breakeven 3ATR vs OFF Trend Breakeven 5/4ATR =====


,exit_profile,strategy_bucket,entry_type,start_time,end_time,entries,units_bought,avg_entry_price,base_entry_price,exit_price,pnl,net_return,exit_reason,entry_fees,exit_fee_tax
0,breakeven,Squeeze OFF bucket,squeeze_off,2016-07-04,2016-11-09,2,31000,12.074797,12.0290,12.906937,23492.442207,0.062671,squeeze_off_breakeven_intraday,533.404148,1770.509120
1,breakeven,Squeeze OFF bucket,squeeze_off,2018-07-16,2018-10-05,1,31000,16.254500,16.2545,16.471127,3737.965390,0.007408,squeeze_off_breakeven_intraday,718.042537,2259.426836
2,breakeven,Squeeze OFF bucket,squeeze_off,2019-09-03,2020-01-30,1,30000,16.745600,16.7456,19.311073,73684.766124,0.146466,squeeze_off_breakeven_intraday,715.874400,2563.544921
3,breakeven,Squeeze OFF bucket,squeeze_off,2020-06-04,2020-06-12,1,21000,18.483300,18.4833,18.430500,-3374.566965,-0.008682,squeeze_off_breakeven_gap_open,553.112753,1712.654212
4,breakeven,Squeeze OFF bucket,squeeze_off,2020-09-17,2021-01-29,1,17000,22.564500,22.5645,27.975195,89330.757328,0.232546,squeeze_off_breakeven_intraday,546.625013,2104.434050
5,breakeven,Squeeze OFF bucket,squeeze_off,2021-03-31,2021-05-04,1,19000,29.524100,29.5241,29.524100,-3281.603715,-0.005842,squeeze_off_breakeven_intraday,799.365008,2482.238708
6,breakeven,Squeeze OFF bucket,squeeze_off,2022-11-10,2022-12-28,1,25000,23.407800,23.4078,24.858799,32691.059884,0.055784,squeeze_off_breakeven_intraday,833.902875,2750.004606
7,breakeven,Squeeze OFF bucket,squeeze_off,2024-11-12,2025-01-13,1,12000,47.130200,47.1302,47.130200,-3308.540040,-0.005842,squeeze_off_breakeven_intraday,805.926420,2502.613620
8,breakeven,Squeeze ON bucket,squeeze_on,2016-06-30,2016-08-22,2,22000,11.861891,11.8561,12.626397,15218.081127,0.058232,squeeze_on_breakeven_intraday,371.870280,1229.179735
9,breakeven,Squeeze ON bucket,squeeze_on,2016-10-24,2016-11-03,2,13000,13.421100,13.4211,13.153253,-4487.272521,-0.025682,squeeze_on_breakeven_intraday,248.625877,756.640902


===== Trade Cycles: Buckets - ON 8/21 EMA vs OFF 13/34 EMA =====


,exit_profile,strategy_bucket,entry_type,start_time,end_time,entries,units_bought,avg_entry_price,base_entry_price,exit_price,pnl,net_return,exit_reason,entry_fees,exit_fee_tax
0,ema_cross,Squeeze OFF bucket,squeeze_off,2016-07-04,2016-11-15,2,31000,12.074797,12.0290,12.8592,22019.135093,0.058741,squeeze_off_ema_cross_next_open_exit,533.404148,1763.960760
1,ema_cross,Squeeze OFF bucket,squeeze_off,2018-07-16,2018-10-09,1,31000,16.254500,16.2545,16.3408,-284.291777,-0.000563,squeeze_off_ema_cross_next_open_exit,718.042537,2241.549240
2,ema_cross,Squeeze OFF bucket,squeeze_off,2019-09-03,2020-02-04,1,30000,16.745600,16.7456,18.8952,61263.787800,0.121776,squeeze_off_ema_cross_next_open_exit,715.874400,2508.337800
3,ema_cross,Squeeze OFF bucket,squeeze_off,2020-06-04,2021-05-13,3,23000,19.140778,18.4833,27.4479,187642.950970,0.425624,squeeze_off_ema_cross_next_open_exit,627.339008,2793.510023
4,ema_cross,Squeeze OFF bucket,squeeze_off,2022-11-10,2022-12-30,1,28000,23.407800,23.4078,25.0256,41263.756940,0.062868,squeeze_off_ema_cross_next_open_exit,933.971220,3100.671840
5,ema_cross,Squeeze OFF bucket,squeeze_off,2024-11-12,2024-11-29,1,14000,47.130200,47.1302,44.6554,-38353.849520,-0.058045,squeeze_off_ema_cross_next_open_exit,940.247490,2766.402030
6,ema_cross,Squeeze ON bucket,squeeze_on,2016-06-30,2016-09-14,2,22000,11.861891,11.8561,12.6197,15071.401925,0.057671,squeeze_on_ema_cross_next_open_exit,371.870280,1228.527795
7,ema_cross,Squeeze ON bucket,squeeze_on,2016-10-24,2016-11-07,2,13000,13.421100,13.4211,13.2276,-3525.043567,-0.020175,squeeze_on_ema_cross_next_open_exit,248.625877,760.917690
8,ema_cross,Squeeze ON bucket,squeeze_on,2017-09-21,2017-09-25,2,17000,15.614059,15.6079,15.5128,-3266.600955,-0.012289,squeeze_on_ema_cross_next_open_exit,378.250575,1166.950380
9,ema_cross,Squeeze ON bucket,squeeze_on,2017-11-13,2017-11-17,2,17000,16.026965,16.0264,16.0360,-1440.961320,-0.005281,squeeze_on_ema_cross_next_open_exit,388.253220,1206.308100


C:\Users\user\AppData\Local\Temp\ipykernel_46272\3650255287.py:725: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
